# Guided Lab: MCP Core Concepts and a LangChain Agent

Use case: a public-policy research assistant that uses a real PDF, a local MCP server, and a LangChain agent.

What you will build:
- an MCP server with `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`
- a stdio transport server for local development
- a LangChain agent that loads MCP tools, resources, and prompts
- a structured-output schema for the final answer
- a practical policy-research workflow backed by a real PDF

MCP standardizes how hosts, clients, and servers exchange tools and context. LangChain’s MCP docs say agents can use tools defined on MCP servers through `langchain-mcp-adapters`, and the MCP architecture docs describe the host/client/server split. The MCP SDK quick example shows `FastMCP` with tool, resource, and prompt decorators, and the inspector docs recommend MCP Inspector for debugging servers. citeturn861028search7turn861028search2turn972229view0turn858732search10

## Core concepts

- **Host**: the app users interact with, such as this notebook or a chat app.
- **Client**: the protocol component inside the host that connects to MCP servers.
- **Server**: the program exposing tools, resources, and prompts.
- **Transport**: the channel used for communication, such as `stdio` or HTTP-based transports.

LangChain’s MCP docs say `stdio` is best for local tools and simple setups, while HTTP connections can pass headers and support both `sse` and `streamable_http` transports. The MCP transport spec shows that HTTP-based transports use POST for JSON-RPC messages and may open SSE streams for server messages; the newer spec page also documents stateful sessions and compatibility with older SSE-style transport. citeturn370462view0turn163347view2turn427329view3turn861028search8turn861028search12

## Learning goals

By the end of this notebook, you should be able to:

1. Explain what MCP solves.
2. Build a small MCP server.
3. Expose a tool, resource, and prompt.
4. Connect the server to a LangChain agent.
5. Understand schema validation for tool inputs and final structured output.
6. Inspect the server with MCP Inspector.

## 1) Install packages

In [ ]:
%pip install -qU     python-dotenv requests pypdf     langchain langchain-core langchain-groq langchain-mcp-adapters     langchain-chroma chromadb sentence-transformers     mcp

## 2) Load environment variables

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=mcp_policy_research_lab
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
LANGSMITH_TRACING = os.getenv('LANGSMITH_TRACING', 'true').lower() == 'true'
LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', 'mcp_policy_research_lab')

print('GROQ_API_KEY set:', bool(GROQ_API_KEY))
print('LANGSMITH_API_KEY set:', bool(LANGSMITH_API_KEY))
print('LANGSMITH_TRACING:', LANGSMITH_TRACING)
print('LANGSMITH_PROJECT:', LANGSMITH_PROJECT)

## 3) Download a real PDF

We will use the public-policy paper **Explainable Machine Learning for Public Policy: Use Cases, Gaps, and Research Directions** from arXiv. It is a good fit because it contains a concrete policy use case, research gaps, and discussion suitable for retrieval and summarization. citeturn360651search0turn360651search3turn360651search12

In [ ]:
from pathlib import Path
import requests

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

PDF_URL = 'https://www.arxiv.org/pdf/2010.14374.pdf'
PDF_PATH = DATA_DIR / 'public_policy_explainability.pdf'

def download_file(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print('Using cached file:', dest)
        return dest
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print('Downloaded:', dest)
    return dest

download_file(PDF_URL, PDF_PATH)

## 4) Load and chunk the PDF

LangChain’s knowledge-base flow is load → split → embed → store. We use that pattern here so the MCP tool has real document content to search. citeturn360651search13

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(pages)

print('Pages:', len(pages))
print('Chunks:', len(chunks))
print('First page preview:')
print(pages[0].page_content[:500])

## 5) Build a local Chroma index

The Chroma index powers the semantic search tool. LangChain provides Chroma integration for local vector search. citeturn360651search1turn360651search4

In [ ]:
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = Path('./chroma_policy_store')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='public_policy_explainability',
    persist_directory=str(CHROMA_DIR),
)

retriever = vector_store.as_retriever(search_kwargs={'k': 4})

print('Chroma store ready at', CHROMA_DIR.resolve())

## 6) Create the local SQLite store

SQLite stores structured data such as metadata, notes, and query history.

In [ ]:
import sqlite3
from datetime import datetime

SQLITE_DB = DATA_DIR / 'policy_assistant.db'
if SQLITE_DB.exists():
    SQLITE_DB.unlink()

conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

cur.execute('CREATE TABLE paper_metadata (paper_id INTEGER PRIMARY KEY AUTOINCREMENT, title TEXT NOT NULL, authors TEXT NOT NULL, year INTEGER NOT NULL, pdf_url TEXT NOT NULL, local_path TEXT NOT NULL, description TEXT NOT NULL)')
cur.execute('CREATE TABLE paper_notes (note_id INTEGER PRIMARY KEY AUTOINCREMENT, page_number INTEGER NOT NULL, note_text TEXT NOT NULL, created_at TEXT NOT NULL)')
cur.execute('CREATE TABLE query_log (log_id INTEGER PRIMARY KEY AUTOINCREMENT, question TEXT NOT NULL, intent TEXT NOT NULL, source TEXT NOT NULL, created_at TEXT NOT NULL)')

cur.execute(
    'INSERT INTO paper_metadata (title, authors, year, pdf_url, local_path, description) VALUES (?, ?, ?, ?, ?, ?)',
    (
        'Explainable Machine Learning for Public Policy: Use Cases, Gaps, and Research Directions',
        'Kasun Amarasinghe; Kit Rodolfa; Hemank Lamba; Rayid Ghani',
        2020,
        PDF_URL,
        str(PDF_PATH),
        'A public-policy research paper about explainability use cases, user goals, and research gaps.'
    )
)

for page in chunks[:8]:
    cur.execute(
        'INSERT INTO paper_notes (page_number, note_text, created_at) VALUES (?, ?, ?)',
        (
            int(page.metadata.get('page', 0)),
            page.page_content[:600].replace('\n', ' '),
            datetime.utcnow().isoformat(),
        )
    )

conn.commit()
conn.close()

print('SQLite database created:', SQLITE_DB.resolve())

## 7) Inspect the structured data

In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

print('paper_metadata rows:')
for row in cur.execute('SELECT title, authors, year FROM paper_metadata'):
    print(row)

print('\npaper_notes sample rows:')
for row in cur.execute('SELECT page_number, substr(note_text, 1, 120) FROM paper_notes LIMIT 3'):
    print(row)

conn.close()

## 8) Write the MCP server

The MCP SDK quick example shows that `FastMCP` can expose a tool, resource, and prompt. The Python SDK docs say type hints and docstrings generate tool definitions automatically, which gives you schema validation on the tool input side. citeturn972229view0turn858732search7

In [ ]:
SERVER_PATH = Path('./policy_mcp_server.py')

server_lines = [
    'from pathlib import Path',
    'import sqlite3',
    '',
    'from mcp.server.fastmcp import FastMCP',
    'from langchain_huggingface import HuggingFaceEmbeddings',
    'from langchain_chroma import Chroma',
    '',
    'BASE_DIR = Path(".")',
    'CHROMA_DIR = BASE_DIR / "chroma_policy_store"',
    'SQLITE_DB = BASE_DIR / "policy_assistant.db"',
    '',
    'mcp = FastMCP("policy-research", json_response=True)',
    '',
    'EMBEDDINGS = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")',
    'STORE = Chroma(collection_name="public_policy_explainability", persist_directory=str(CHROMA_DIR), embedding_function=EMBEDDINGS)',
    '',
    'def _paper_metadata() -> str:',
    '    conn = sqlite3.connect(SQLITE_DB)',
    '    cur = conn.cursor()',
    '    row = cur.execute("SELECT title, authors, year, description FROM paper_metadata LIMIT 1").fetchone()',
    '    conn.close()',
    '    if not row:',
    '        return "No metadata available."',
    '    return f"Title: {row[0]} | Authors: {row[1]} | Year: {row[2]} | Description: {row[3]}"',
    '',
    '@mcp.tool()',
    'def search_policy(query: str, k: int = 3) -> str:',
    '    "Search the policy paper for relevant passages."',
    '    docs = STORE.similarity_search(query, k=k)',
    '    lines = []',
    '    for doc in docs:',
    '        page = doc.metadata.get("page", "n/a")',
    '        snippet = doc.page_content[:700].replace("\n", " ")',
    '        lines.append(f"[page={page}] {snippet}")',
    '    return "\n\n".join(lines)',
    '',
    '@mcp.resource("policy://metadata")',
    'def metadata_resource() -> str:',
    '    "Expose the paper metadata as a readable resource."',
    '    return _paper_metadata()',
    '',
    '@mcp.resource("policy://note/{note_id}")',
    'def note_resource(note_id: str) -> str:',
    '    "Expose a saved note from SQLite."',
    '    conn = sqlite3.connect(SQLITE_DB)',
    '    cur = conn.cursor()',
    '    row = cur.execute("SELECT page_number, note_text FROM paper_notes WHERE note_id = ?", (note_id,)).fetchone()',
    '    conn.close()',
    '    if not row:',
    '        return f"No note found for id={note_id}"',
    '    return f"Page {row[0]}: {row[1]}"',
    '',
    '@mcp.prompt()',
    'def policy_brief(topic: str, audience: str = "policy analyst") -> str:',
    '    "Return a reusable policy brief prompt template."',
    '    return (',
    '        f"Write a concise policy brief for {audience}.\n\n"',
    '        f"Topic: {topic}\n\n"',
    '        "Use only the evidence from the policy paper. "',
    '        "Include: Overview, Evidence, Gaps, and Recommendation."',
    '    )',
    '',
    'if __name__ == "__main__":',
    '    mcp.run(transport="stdio")',
]

SERVER_PATH.write_text("\n".join(server_lines) + "\n", encoding='utf-8')
print('Wrote MCP server to', SERVER_PATH.resolve())

## 9) MCP transports

Use **stdio** for local development. LangChain says `stdio` is best for local tools and simple setups. For remote HTTP servers, LangChain supports `sse` and `streamable_http`; the MCP transport docs and LangChain docs note that SSE is the legacy/deprecated style and that HTTP-based transports carry JSON-RPC over POST plus SSE/event streams where supported. citeturn370462view0turn163347view2turn427329view3

In [ ]:
# stdio (local, recommended for this lab)
stdio_config = {
    'policy': {
        'transport': 'stdio',
        'command': 'python',
        'args': [str(SERVER_PATH)],
    }
}

# legacy SSE example (for compatibility only)
# sse_config = {
#     'policy': {
#         'transport': 'sse',
#         'url': 'http://localhost:8000/sse',
#     }
# }

stdio_config

## 10) MCP Inspector

The MCP Inspector is the first-stop debugging tool for MCP servers. It can connect to stdio or Streamable HTTP servers and lets you invoke tools, prompts, and resources interactively. citeturn427329view2turn427329view4

In [ ]:
print('To inspect the stdio server:')
print('npx -y @modelcontextprotocol/inspector python policy_mcp_server.py')
print()
print('To inspect a remote HTTP server:')
print('npx -y @modelcontextprotocol/inspector http://localhost:8000/mcp')

## 11) Connect MCP tools, resources, and prompts with LangChain

LangChain says `MultiServerMCPClient` can load tools, resources, and prompts from one or more MCP servers. It is stateless by default, and you can use `client.session()` when you want a persistent session. citeturn163347view3turn163347view4turn271895view0turn271895view1turn271895view2

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

class PolicyBrief(BaseModel):
    title: str = Field(description='Brief title')
    bullets: list[str] = Field(description='Supporting bullets grounded in the document')
    recommendation: str = Field(description='Short recommendation')

GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.3-70b-versatile')
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

async def load_mcp_components():
    client = MultiServerMCPClient(stdio_config)

    tools = await client.get_tools()
    blobs = await client.get_resources('policy', uris=['policy://metadata'])
    prompt_messages = await client.get_prompt(
        'policy',
        'policy_brief',
        arguments={'topic': 'explainable machine learning for public policy', 'audience': 'policy analyst'}
    )

    return client, tools, blobs, prompt_messages

client, tools, blobs, prompt_messages = await load_mcp_components()

print('Loaded MCP tools:', [t.name for t in tools])
print('Loaded resource preview:')
for blob in blobs:
    print(blob.as_string())
print()
print('Loaded prompt messages:')
for message in prompt_messages:
    print(message.type, message.content)

## 12) Build the LangChain agent on top of MCP tools

LangChain’s agent docs say `create_agent` is the standard agent harness, and the v1 notes say it runs on LangGraph. We use it here so the MCP tools become ordinary agent tools. citeturn503993search0turn503993search6turn163347view3

In [ ]:
agent = create_agent(
    llm,
    tools,
    system_prompt=(
        'You are a public-policy research assistant. Use MCP tools to retrieve evidence and resources. '
        'Always ground your answer in the document and keep it concise.'
    ),
    response_format=PolicyBrief,
)

print('Agent ready.')

## 13) Run a grounded question

This question should use the MCP search tool and return a validated structured response.

In [ ]:
result = await agent.ainvoke({
    'messages': [
        {
            'role': 'user',
            'content': 'What use cases and research gaps does the paper identify for explainable machine learning in public policy?',
        }
    ]
})

result

## 14) Inspect the structured response

In [ ]:
structured = result['structured_response']
structured

## 15) Why this validates schemas well

On the server side, MCP tool schemas are generated from Python type hints and docstrings in `FastMCP`. On the agent side, LangChain’s structured output validates the final answer schema and can produce clear errors if the output does not match. citeturn972229view0turn427329view5

## 16) Example of a direct resource and prompt workflow

Resources are for read-only context, while prompts are reusable templates that the client can load and then feed into a workflow. LangChain converts MCP prompts into messages, which makes them easy to reuse in chat-style flows. citeturn271895view0turn271895view1turn858732search5

In [ ]:
for message in prompt_messages:
    print(message.type, ':', message.content)

## 17) What this notebook demonstrates

- MCP solves the host/client/server integration problem for LLM apps.
- `stdio` is the simplest transport for local development.
- SSE appears as legacy compatibility in the current docs, while Streamable HTTP is the modern HTTP transport.
- `FastMCP` gives you `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`.
- LangChain’s MCP adapters let a LangChain agent consume those server capabilities directly.

## Key takeaways

The practical pattern is: keep the source document in a local semantic store, expose it through MCP as tools/resources/prompts, and let a LangChain agent consume those server capabilities. That gives you a clean separation between the host, the client connection, the MCP server, and the transport layer. citeturn861028search7turn861028search2turn858732search0turn163347view3turn370462view0turn427329view4

## References

- MCP architecture overview: https://modelcontextprotocol.io/docs/learn/architecture
- MCP clients: https://modelcontextprotocol.io/docs/learn/client-concepts
- MCP servers: https://modelcontextprotocol.io/docs/learn/server-concepts
- MCP Inspector: https://modelcontextprotocol.io/docs/tools/inspector
- Build an MCP server: https://modelcontextprotocol.io/docs/develop/build-server
- LangChain MCP integration: https://docs.langchain.com/oss/python/langchain/mcp
- LangChain MCP tools: https://docs.langchain.com/oss/python/deepagents/code/mcp-tools